# F15 — Empaquetado de shards DL sobre Colab (cómputo fuera del PC local)

Política del equipo (13-oct): el cómputo pesado corre en Colab; el repo local
solo recibe los artefactos ligeros ya verificados. Este notebook provisiona
el dato crudo ZZU (~4 GB comprimido), ejecuta `scripts/empaquetar_shards_dl.py`
(int16, shards ~1 GB, splits 80/10/10 por paciente, stats de train) y sube
los shards a Drive para F16–F18.

Las celdas 1–2 (SSH) permiten que el asistente conduzca el resto por SSH
(`ssh colab-f15`); si prefieres la UI de Colab, ejecuta de arriba a abajo.

## 1. Setup SSH (igual que colab_f14)

In [1]:
%%bash
apt-get -qq update >/dev/null && apt-get -qq install -y openssh-server p7zip-full >/dev/null
mkdir -p /run/sshd /root/.ssh && chmod 700 /root/.ssh
sed -i 's/#\?Port .*/Port 22/' /etc/ssh/sshd_config
nohup /usr/sbin/sshd >/dev/null 2>&1 &
echo "sshd arriba"

sshd arriba


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
PUBKEY = "ssh-ed25519 AAAA... tu@pc"
TS_AUTHKEY = "tskey-auth-..."

open("/root/.ssh/authorized_keys", "w").write(PUBKEY + "\n")
import os; os.chmod("/root/.ssh/authorized_keys", 0o600)
!curl -fsSL https://tailscale.com/install.sh | sh
!tailscale up --hostname colab-f15 --authkey="$TS_AUTHKEY" --ssh=false
!tailscale ip -4

Installing Tailscale for ubuntu noble, using method apt
+ mkdir -p --mode=0755 /usr/share/keyrings
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/noble.noarmor.gpg
+ tee /usr/share/keyrings/tailscale-archive-keyring.gpg
+ chmod 0644 /usr/share/keyrings/tailscale-archive-keyring.gpg
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/noble.tailscale-keyring.list
+ tee /etc/apt/sources.list.d/tailscale.list
# Tailscale packages for ubuntu noble
deb [signed-by=/usr/share/keyrings/tailscale-archive-keyring.gpg] https://pkgs.tailscale.com/stable/ubuntu noble main
+ chmod 0644 /etc/apt/sources.list.d/tailscale.list
+ apt-get update
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease     
Get:3 https://pkgs.tailscale.com/stable/ubuntu noble InRelease                 
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease               
Hit:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease    

## 2. Repo + dependencias + dato crudo ZZU

In [3]:
!test -d Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG || git clone --depth 1 "$REPO" -b etapa2-corpus-combinado
%cd Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG
# sync robusto a clones shallow de otra rama: fetch explícito + checkout de FETCH_HEAD
!git fetch origin etapa2-corpus-combinado --depth 1
!git checkout -B etapa2-corpus-combinado FETCH_HEAD
!git log --oneline -1
!pip install -q wfdb pywavelets
import wfdb, pywt, sklearn, pandas, numpy
print('deps OK')

/content
/content/Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG
d21bf5c (grafted, HEAD -> resultados-colab, origin/main, origin/HEAD, main) Merge busqueda-datos a main: ampliacion de datos ZZU pECG (AP-13) + Etapa I-bis completa + F6-F8 verificados y F9 WIP de desarrollo
deps OK


In [4]:
# ZZU pECG multipartito con verificación MD5 (docs/metodologia_zzu_pecg.md §1)
ARCHIVOS = {
    'Child_ecg.z01': ('https://ndownloader.figshare.com/files/53524583', '4782e8c17b478745853db85210ce46aa'),
    'Child_ecg.zip': ('https://ndownloader.figshare.com/files/53523902', 'd7bb207d0f559c5f697bd7e442975718'),
}
import hashlib, pathlib, subprocess
dl = pathlib.Path('/content/descargas'); dl.mkdir(exist_ok=True)
for nombre, (url, md5) in ARCHIVOS.items():
    destino = dl / nombre
    if not destino.exists():
        subprocess.run(['curl', '-L', '-C', '-', '-o', str(destino), url], check=True)
    h = hashlib.md5(destino.read_bytes()).hexdigest()
    assert h == md5, f'{nombre}: MD5 {h} != {md5}'
    print(nombre, 'MD5 OK')
data = pathlib.Path('data/zzu_pecg'); data.mkdir(parents=True, exist_ok=True)
if not (data / 'Child_ecg').exists():
    subprocess.run(['7z', 'x', f'-o{data}', str(dl / 'Child_ecg.zip')], check=True)
n_hea = len(list((data / 'Child_ecg').rglob('*.hea')))
assert n_hea == 14190, n_hea
print(f'Child_ecg: {n_hea} registros')

Child_ecg.z01 MD5 OK
Child_ecg.zip MD5 OK
Child_ecg: 14190 registros


## 3. F15 — empaquetado + verificación (~40–60 min en runtime CPU)

In [5]:
!python scripts/empaquetar_shards_dl.py
!cat results/dl/verificacion_shards.txt

python3: can't open file '/content/Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG/scripts/empaquetar_shards_dl.py': [Errno 2] No such file or directory
cat: results/dl/verificacion_shards.txt: No such file or directory


## 4. Persistencia: shards a Drive, manifiesto a git

In [6]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/arritmias_dl/shards
!cp -n results/dl/shards/*.npy /content/drive/MyDrive/arritmias_dl/shards/
!cp results/dl/{manifiesto,splits}.csv results/dl/{normalizacion.json,verificacion_shards.txt} /content/drive/MyDrive/arritmias_dl/
!ls -lh /content/drive/MyDrive/arritmias_dl/shards | head
# Artefactos ligeros al repo (rama etapa2-corpus-combinado):
!git config user.email "colab@runtime" && git config user.name "colab"
!git add results/dl/manifiesto.csv results/dl/splits.csv results/dl/normalizacion.json results/dl/verificacion_shards.txt
!git commit -m "F15: shards DL empaquetados en Colab (verificacion VERDE)" && git push origin etapa2-corpus-combinado

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
cp: cannot stat 'results/dl/shards/*.npy': No such file or directory
cp: cannot stat 'results/dl/manifiesto.csv': No such file or directory
cp: cannot stat 'results/dl/splits.csv': No such file or directory
cp: cannot stat 'results/dl/normalizacion.json': No such file or directory
cp: cannot stat 'results/dl/verificacion_shards.txt': No such file or directory
total 0
fatal: pathspec 'results/dl/manifiesto.csv' did not match any files
On branch resultados-colab
nothing to commit, working tree clean
